# Módulo 2: Modelado, Baseline y Evaluación
## Proyecto MLOps - CarDekho Resale Price Prediction

### 1. Justificación Problema -> Tarea -> Métrica (M2.1)
* **Problema:** Estimar el precio justo de mercado de un coche usado según sus características técnicas y de uso.
* **Tarea:** Regresión Supervisada sobre `selling_price`.
* **Métricas Elegidas:**
  - **MAE (Mean Absolute Error):** Métrica principal de negocio. Mide la desviación media en rupias indias sin distorsionar la interpretación humana.
  - **RMSE (Root Mean Squared Error):** Penaliza desviaciones cuadráticas grandes en coches de alta gama.
  - **R² (Coeficiente de Determinación):** Cuantifica el porcentaje de varianza del precio explicada por el modelo.

### 2. Estrategia de Splits sin Fugas (M2.3)
* Particionado en **Train (70%)**, **Validation (15%)** y **Test (15%)** con `random_state=42`.
* El preprocesamiento (`StandardScaler` y `OneHotEncoder`) se ajusta exclusivamente con el conjunto de entrenamiento.
* Las decisiones de selección y comparativa se realizan sobre el conjunto de Validación, dejando Test completamente sellado para la evaluación final.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
%matplotlib inline

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

features = ["year", "km_driven", "fuel", "seller_type", "transmission", "owner", "brand"]
target = "selling_price"

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Dimensiones: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

### 3. Pipeline de Preprocesamiento y Comparación con Dummy Baseline (M2.2)

In [ ]:
cat_cols = ["brand", "fuel", "seller_type", "transmission", "owner"]
num_cols = ["year", "km_driven"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

# Comparación de modelos con seed fija para reproducibilidad (M2.5)
models = {
    "Dummy (Media)": DummyRegressor(strategy="mean"),
    "Ridge": Pipeline([("prep", preprocessor), ("m", Ridge(alpha=1.0, random_state=42))]),
    "Random Forest": Pipeline([("prep", preprocessor), ("m", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("prep", preprocessor), ("m", GradientBoostingRegressor(n_estimators=100, random_state=42))])
}

records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    r2 = r2_score(y_val, preds)
    records.append({"Modelo": name, "Val MAE": mae, "Val RMSE": rmse, "Val R²": r2})

results_df = pd.DataFrame(records)
results_df

### 4. Evaluación del Modelo Campeón en Test Set y Visualizaciones (M2.4)

In [ ]:
champion = models["Gradient Boosting"]
test_preds = champion.predict(X_test)
residuals = y_test - test_preds

print(f"Test MAE:  {mean_absolute_error(y_test, test_preds):,.0f} rupias")
print(f"Test RMSE: {root_mean_squared_error(y_test, test_preds):,.0f} rupias")
print(f"Test R²:   {r2_score(y_test, test_preds):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Residuales vs Predicción
axes[0].scatter(test_preds, residuals, alpha=0.5, color="royalblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuales vs Predicción (Test Set)")
axes[0].set_xlabel("Predicciones")
axes[0].set_ylabel("Residuales")

# Valores Reales vs Predichos
axes[1].scatter(y_test, test_preds, alpha=0.5, color="darkgreen")
diag = np.linspace(min(y_test.min(), test_preds.min()), max(y_test.max(), test_preds.max()), 100)
axes[1].plot(diag, diag, color="red", linestyle="--", label="1:1 Ideal")
axes[1].set_title("Valores Reales vs Predichos")
axes[1].set_xlabel("Valores Reales")
axes[1].set_ylabel("Predicciones")
axes[1].legend()

plt.tight_layout()
plt.show()